<a href="https://colab.research.google.com/github/canevadop-dev/proyecto-datamining/blob/main/Primera_entrega_nueva_fuente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


---
# Proyecto: Data Mining
## Integración de fuentes: Matrícula/Trayectoria, Deserción y Padrón de IIEE


**Problema:** La deserción escolar constituye uno de los desafíos estructurales más críticos del sistema educativo peruano, afectando directamente las trayectorias de desarrollo socioeconómico y profundizando las brechas de equidad territorial.


**Objetivo:** integrar cuatro fuentes públicas del sector Educación (Perú) en una sola base
de análisis a nivel **distrito x nivel educativo**, usando el **Padrón de Instituciones
Educativas (`Padron.dbf`)** como tabla puente entre la matrícula (a nivel de escuela) y la
deserción (a nivel de distrito/ubigeo).

**Análisis:** Comparar la deserción entre niveles educativos, medir brechas distritales y evaluar si indicadores de trayectoría escolar anticipan la deserción.

**Fuentes:**
1. `Padron.dbf` — catálogo de IIEE. Puente: da la ubicación geográfica (`ubigeo`/`CODGEO`) de cada
   escuela (`COD_MOD` + `ANEXO`).
2. `Matriculación_y_Trayectoria_Estudiantil_2023.csv` — matrícula y trayectoria por escuela y nivel, año 2023. Se filtra a **Primaria** y **Secundaria**.
3. `Tasa_y_número_de_desertores_en_Educación_Primaria_2023-2024.csv` — nivel Primaria.
4. `Tasa_y_número_de_desertores_en_Educación_Secundaria_2023-2024.csv` — nivel Secundaria.

**Definición importante de las fuentes 3 y 4 (deserción):** no son un conteo independiente por
año; es una **medida de cohorte**: `denominador` = estudiantes matriculados en **2023**, y
`desertor` = de ese grupo, cuántos **no volvieron a matricularse en 2024** (en cualquier escuela
del sistema). `Tasa` = `desertor / denominador`. Esto implica que la deserción debe cruzarse
contra la **matrícula 2023** (que es su propio denominador).

**Unidad de análisis:** Distrito: cada fila de la base final representa un distrito individual (identificado de manera única por su código UBIGEO), registrando simultáneamente sus métricas de deserción en primaria y secundaria.

---


# <font color=green>Parte 1. Cargar e inspeccionar las fuentes</font>
Antes de integrar, debemos entender qué contiene cada fuente.

In [ ]:
!pip install dbfread pandas
import pandas as pd
import numpy as np
import plotly.express as px
from dbfread import DBF


RUTA = "/content/"


### 1.0 Carga de las 4 fuentes

In [ ]:

# Fuente 1: Padrón (puente) ---
# Codepage cp850: el .dbf viene con codepage ID=0x2 (DOS Latin-1 / cp850),
# necesario para que tildes y eñes se lean correctamente.
cols_padron = ['COD_MOD','ANEXO','NIV_MOD','D_NIV_MOD','CODGEO','DPTO','PROV','DIST',
               'D_GESTION','D_COD_TUR','CEN_EDU','DRE_UGEL','TALUMNO','AREA_CENSO','DAREACENSO']
tabla = DBF(RUTA + "Padron.dbf", encoding="cp850", load=False)
padron = pd.DataFrame([{c: rec[c] for c in cols_padron} for rec in tabla])

print("Fuente 1: padron")
display(padron.head())


Fuente 1: padron


,COD_MOD,ANEXO,NIV_MOD,D_NIV_MOD,CODGEO,DPTO,PROV,DIST,D_GESTION,D_COD_TUR,CEN_EDU,DRE_UGEL,TALUMNO,AREA_CENSO,DAREACENSO
0,0415547,0,A2,Inicial - Jardín,020105,ANCASH,HUARAZ,INDEPENDENCIA,Pública de gestión directa,Mañana,123,UGEL HUARAZ,436,1,Urbana
1,0415638,0,A2,Inicial - Jardín,020101,ANCASH,HUARAZ,HUARAZ,Pública de gestión directa,Mañana,122,UGEL HUARAZ,481,1,Urbana
2,0415646,0,A2,Inicial - Jardín,020101,ANCASH,HUARAZ,HUARAZ,Pública de gestión directa,Mañana-Tarde,233,UGEL HUARAZ,555,1,Urbana
3,0415877,0,A2,Inicial - Jardín,020105,ANCASH,HUARAZ,INDEPENDENCIA,Privada,Mañana-Tarde,COLEGIO PARROQUIAL NUESTRA SEÑORA DEL SAGRADO ...,UGEL HUARAZ,233,1,Urbana
4,0567206,0,A2,Inicial - Jardín,020105,ANCASH,HUARAZ,INDEPENDENCIA,Pública de gestión directa,Mañana,268,UGEL HUARAZ,64,2,Rural


In [ ]:

# Fuente 2: Matrícula y Trayectoria 2023
matricula_2023 = pd.read_csv(RUTA + "Matriculación y Trayectoria Estudiantil 2023.csv")

print("Fuente 2: matricula_2023")
display(matricula_2023.head())


Fuente 2: matricula_2023


,cod_mod,anexo,Nombre,gestion,id_nivel,dsc_nivel,Edad,TipoDiscaIntegrada,TotalEstudiantes,Discapacidad,...,DNI_SinValidar,No_DNI,Aprobado,Desaprobado,Retirado,Fallecido,RequiereRecuperacion,Matriculado,PostergaEvaluacion,tot_atraso
0,1506,0,LOS LUCEROS,Pública de gestión directa,A5,Inicial no escolarizado,3,NaN,2,0,...,0,0,2,0,0,0,0,0,0,0
1,1506,0,LOS LUCEROS,Pública de gestión directa,A5,Inicial no escolarizado,4,NaN,2,0,...,0,0,2,0,0,0,0,0,0,0
2,1506,0,LOS LUCEROS,Pública de gestión directa,A5,Inicial no escolarizado,5,NaN,1,0,...,0,0,1,0,0,0,0,0,0,0
3,1507,0,EL NUEVO AMANECER,Pública de gestión directa,A5,Inicial no escolarizado,3,NaN,5,0,...,0,0,5,0,0,0,0,0,0,0
4,1507,0,EL NUEVO AMANECER,Pública de gestión directa,A5,Inicial no escolarizado,4,NaN,1,0,...,0,0,1,0,0,0,0,0,0,0


In [ ]:

# Fuentes 3 y 4: Deserción Primaria / Secundaria (a nivel distrito, cohorte 2023-2024)
desercion_primaria = pd.read_csv(RUTA + "Tasa y número de desertores en Educación Primaria 2023-2024.csv")
desercion_secundaria = pd.read_csv(RUTA + "Tasa y número de desertores en Educación Secundaria 2023-2024.csv")

print("Fuente 3: desercion_primaria")
display(desercion_primaria.head())

print("Fuente 4: desercion_secundaria")
display(desercion_secundaria.head())


Fuente 3: desercion_primaria


,ubigeo,Departamento,Provincia,Distrito,desertor,denominador,Tasa
0,10101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,17,4146,0.410034
1,10102,AMAZONAS,CHACHAPOYAS,ASUNCION,0,12,0.000000
2,10103,AMAZONAS,CHACHAPOYAS,BALSAS,0,168,0.000000
3,10104,AMAZONAS,CHACHAPOYAS,CHETO,0,44,0.000000
4,10105,AMAZONAS,CHACHAPOYAS,CHILIQUIN,3,62,4.838710


Fuente 4: desercion_secundaria


,ubigeo,Departamento,Provincia,Distrito,desertor,denominador,Tasa
0,10101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,42,3421,1.227711
1,10103,AMAZONAS,CHACHAPOYAS,BALSAS,5,125,4.000000
2,10104,AMAZONAS,CHACHAPOYAS,CHETO,0,54,0.000000
3,10106,AMAZONAS,CHACHAPOYAS,CHUQUIBAMBA,7,171,4.093567
4,10107,AMAZONAS,CHACHAPOYAS,GRANADA,0,62,0.000000


**Dimensiones de cada fuente de datos**

In [ ]:

import pandas as pd

# 1. Crear una lista con la información de cada DataFrame
datos = []
for nombre, df in [("padron", padron), ("matricula_2023", matricula_2023),
                    ("desercion_primaria", desercion_primaria),
                    ("desercion_secundaria", desercion_secundaria)]:
    # df.shape devuelve una tupla: (filas, columnas)
    datos.append({
        "Fuente": nombre,
        "Filas": df.shape[0],
        "Columnas": df.shape[1],
        "Dimensiones (Filas, Columnas)": str(df.shape)
    })

# 2. Convertir la lista en un DataFrame
df_dimensiones = pd.DataFrame(datos)

# 3. Mostrar la tabla resultante
display(df_dimensiones)

,Fuente,Filas,Columnas,"Dimensiones (Filas, Columnas)"
0,padron,111715,15,"(111715, 15)"
1,matricula_2023,544834,26,"(544834, 26)"
2,desercion_primaria,1890,7,"(1890, 7)"
3,desercion_secundaria,1846,7,"(1846, 7)"


**Tipos de datos**

In [ ]:

import pandas as pd

# 1. Crear una lista para almacenar la información de las columnas y sus tipos
datos_dtypes = []

for nombre, df in [("padron", padron), ("matricula_2023", matricula_2023),
                    ("desercion_primaria", desercion_primaria)]:
    # df.dtypes devuelve una Serie donde el índice es la columna y el valor es el tipo de dato
    df_tipo = df.dtypes.reset_index()
    df_tipo.columns = ["Columna", "Tipo de Dato"] # Renombrar las columnas
    df_tipo["Fuente"] = nombre                    # Agregar una columna con el nombre de la fuente

    # Reordenar las columnas para que "Fuente" quede primero (opcional)
    df_tipo = df_tipo[["Fuente", "Columna", "Tipo de Dato"]]

    datos_dtypes.append(df_tipo)

# 2. Unir todas las tablas en una sola
df_dtypes_global = pd.concat(datos_dtypes, ignore_index=True)

# 3. Mostrar la tabla resultante
display(df_dtypes_global)

,Fuente,Columna,Tipo de Dato
0,padron,COD_MOD,object
1,padron,ANEXO,object
2,padron,NIV_MOD,object
3,padron,D_NIV_MOD,object
4,padron,CODGEO,object
5,padron,DPTO,object
6,padron,PROV,object
7,padron,DIST,object
8,padron,D_GESTION,object
9,padron,D_COD_TUR,object


### <font color=279CF5>1.1 Diagnóstico de valores faltantes por fuente</font>

No hacemos limpieza profunda aún, solo detectamos y registramos el problema.

**Cantidad de valores nulos**

In [ ]:
for nombre, df in [("padron", padron), ("matricula_2023", matricula_2023),
                    ("desercion_primaria", desercion_primaria), ("desercion_secundaria", desercion_secundaria)]:

    print(nombre)
    display(df.isna().sum().reset_index(name="Nulos"))

padron


,index,Nulos
0,COD_MOD,0
1,ANEXO,0
2,NIV_MOD,0
3,D_NIV_MOD,0
4,CODGEO,0
5,DPTO,0
6,PROV,0
7,DIST,0
8,D_GESTION,0
9,D_COD_TUR,0


matricula_2023


,index,Nulos
0,cod_mod,0
1,anexo,0
2,Nombre,0
3,gestion,0
4,id_nivel,0
5,dsc_nivel,0
6,Edad,0
7,TipoDiscaIntegrada,481172
8,TotalEstudiantes,0
9,Discapacidad,0


desercion_primaria


,index,Nulos
0,ubigeo,0
1,Departamento,0
2,Provincia,0
3,Distrito,0
4,desertor,0
5,denominador,0
6,Tasa,0


desercion_secundaria


,index,Nulos
0,ubigeo,0
1,Departamento,0
2,Provincia,0
3,Distrito,0
4,desertor,0
5,denominador,0
6,Tasa,0



**Nota:** en `matricula_2023` la columna `TipoDiscaIntegrada` tiene muchos
nulos porque solo se llena cuando `Discapacidad > 0`; no es un dato faltante real, sino
estructuralmente ausente (categoría "sin discapacidad registrada").


### <font color=279CF5>1.2 Valores distintos en variables categóricas clave</font>

**Análisis de variables categóricas**

In [ ]:

print("Niveles educativos en matrícula (dsc_nivel):")
display(matricula_2023["dsc_nivel"].value_counts(dropna=False))

print("\nTipo de gestión en matrícula:")
display(matricula_2023["gestion"].value_counts(dropna=False))

print("\nNiveles modulares en el padrón (D_NIV_MOD):")
display(padron["D_NIV_MOD"].value_counts(dropna=False))

print("\nGestión en el padrón (D_GESTION):")
display(padron["D_GESTION"].value_counts(dropna=False))


Niveles educativos en matrícula (dsc_nivel):


,count
dsc_nivel,
Primaria,267361
Secundaria,125744
Inicial - Jardín,100659
Inicial no escolarizado,42892
Inicial - Cuna-Jardín,8124
Inicial - Cuna,54



Tipo de gestión en matrícula:


,count
gestion,
Pública de gestión directa,416371
Privada,120024
Pública de gestión privada,8439



Niveles modulares en el padrón (D_NIV_MOD):


,count
D_NIV_MOD,
Primaria,38128
Inicial - Jardín,33100
Inicial - Programa no escolarizado,17751
Secundaria,15214
Técnico Productiva,1606
Inicial - Cuna-jardín,1588
Básica Alternativa-Avanzado,1517
Básica Alternativa-Inicial e Intermedio,853
Instituto Superior Tecnológico,786



Gestión en el padrón (D_GESTION):


,count
D_GESTION,
Pública de gestión directa,85323
Privada,25144
Pública de gestión privada,1248


**Valores duplicados**

In [ ]:
import pandas as pd

for nombre, df in [("padron", padron), ("matricula_2023", matricula_2023),
                    ("desercion_primaria", desercion_primaria),
                    ("desercion_secundaria", desercion_secundaria)]:

    # Calcular duplicados por columna para la fuente actual
    datos_col = []
    for col in df.columns:
        n_dup_col = df[col].duplicated().sum()
        datos_col.append({
            "Columna": col,
            "Valores Duplicados": n_dup_col,
            "Total Filas": len(df)
        })

    # Crear el DataFrame de la fuente
    df_resultado = pd.DataFrame(datos_col)

    # Mostrar título y la tabla correspondiente
    print(f" Duplicados por columna en: {nombre} ")
    display(df_resultado)
    print("\n")

 Duplicados por columna en: padron 


,Columna,Valores Duplicados,Total Filas
0,COD_MOD,11,111715
1,ANEXO,111712,111715
2,NIV_MOD,111698,111715
3,D_NIV_MOD,111698,111715
4,CODGEO,109825,111715
5,DPTO,111690,111715
6,PROV,111519,111715
7,DIST,109981,111715
8,D_GESTION,111712,111715
9,D_COD_TUR,111708,111715




 Duplicados por columna en: matricula_2023 


,Columna,Valores Duplicados,Total Filas
0,cod_mod,439142,544834
1,anexo,544831,544834
2,Nombre,490204,544834
3,gestion,544831,544834
4,id_nivel,544828,544834
5,dsc_nivel,544828,544834
6,Edad,544815,544834
7,TipoDiscaIntegrada,544817,544834
8,TotalEstudiantes,544357,544834
9,Discapacidad,544818,544834




 Duplicados por columna en: desercion_primaria 


,Columna,Valores Duplicados,Total Filas
0,ubigeo,0,1890
1,Departamento,1865,1890
2,Provincia,1694,1890
3,Distrito,156,1890
4,desertor,1694,1890
5,denominador,742,1890
6,Tasa,724,1890




 Duplicados por columna en: desercion_secundaria 


,Columna,Valores Duplicados,Total Filas
0,ubigeo,0,1846
1,Departamento,1821,1846
2,Provincia,1650,1846
3,Distrito,149,1846
4,desertor,1628,1846
5,denominador,769,1846
6,Tasa,612,1846


In [ ]:
# 1.3.1 Duplicados EXACTOS (fila completa) en cada fuente, tal como llegaron
print("Duplicados exactos por fuente:")
for nombre, df in [("padron", padron), ("matricula_2023", matricula_2023),
                    ("desercion_primaria", desercion_primaria),
                    ("desercion_secundaria", desercion_secundaria)]:
    n_dup = df.duplicated().sum()
    print(f"  {nombre}: {n_dup} filas exactamente duplicadas de {len(df)}")

Duplicados exactos por fuente:
  padron: 0 filas exactamente duplicadas de 111715
  matricula_2023: 0 filas exactamente duplicadas de 544834
  desercion_primaria: 0 filas exactamente duplicadas de 1890
  desercion_secundaria: 0 filas exactamente duplicadas de 1846


**Duplicados exactos (fila completa) en cada fila**

In [ ]:
import pandas as pd

# 1. Recopilar los duplicados exactos por fuente
datos_resumen = []

for nombre, df in [("padron", padron), ("matricula_2023", matricula_2023),
                    ("desercion_primaria", desercion_primaria),
                    ("desercion_secundaria", desercion_secundaria)]:

    n_dup = df.duplicated().sum()

    datos_resumen.append({
        "Fuente": nombre,
        "Filas Duplicadas Exactas": n_dup,
        "Total de Filas": len(df)
    })

#  Convertir a DataFrame global
df_duplicados_exactos = pd.DataFrame(datos_resumen)

# Mostrar la tabla con formato claro y separador de miles
display(
    df_duplicados_exactos.style
    .format({"Filas Duplicadas Exactas": "{:,}", "Total de Filas": "{:,}"})
    .hide(axis="index")
)

Fuente,Filas Duplicadas Exactas,Total de Filas
padron,0,"111,715"
matricula_2023,0,"544,834"
desercion_primaria,0,"1,890"
desercion_secundaria,0,"1,846"



# <font color=green>Parte 2. Organizar datos antes de integrar</font>

Necesitamos:
1. Filtrar la matrícula a **Primaria** y **Secundaria** (para que sea comparable con los archivos
   de deserción, que solo cubren esos dos niveles).
2. Normalizar las llaves de unión (`cod_mod`, `anexo`, `ubigeo`) para que tengan el mismo formato
   en todas las fuentes.


**2.1 Filtrar matrícula a Primaria y Secundaria**

In [ ]:
import pandas as pd

# 1. Filtrar solo Primaria y Secundaria
matricula = matricula_2023[matricula_2023["dsc_nivel"].isin(["Primaria", "Secundaria"])].copy()

# 2. Crear una tabla simple para comparar tamaños
df_resultado = pd.DataFrame({
    "Estado": ["Original", "Filtrada"],
    "Total Filas": [len(matricula_2023), len(matricula)]
})

# 3. Mostrar la tabla
display(df_resultado)

,Estado,Total Filas
0,Original,544834
1,Filtrada,393105


**2.2 Adecuación de las variables llaves para el merge (Patrón y matricula)**

Código del colegio (cod_mod): Convierte el código modular a texto y completa con ceros a la izquierda hasta obtener 7 dígitos (.str.zfill(7)), asegurando que códigos como 0123456 y 123456 puedan coincidir correctamente.

Anexo (anexo): Convierte los valores del anexo a texto en ambas tablas, permitiendo que tengan el mismo formato al momento de realizar el cruce de datos.

Código de ubicación geográfica (ubigeo / CODGEO): Convierte los códigos a texto y completa con ceros a la izquierda hasta obtener 6 dígitos (.str.zfill(6)), evitando la pérdida de ceros iniciales al leerlos como números.

In [ ]:

#  - cod_mod en matrícula es numérico (int64); en el padrón es texto con cero a la izquierda (7 dígitos)
#  - anexo en ambas fuentes es numérico/entero, pero en el padrón viene como texto
#  - CODGEO / ubigeo: 6 dígitos con cero a la izquierda

matricula["cod_mod_key"] = matricula["cod_mod"].astype(str).str.zfill(7)
matricula["anexo_key"] = matricula["anexo"].astype(str)

padron["cod_mod_key"] = padron["COD_MOD"].astype(str).str.zfill(7)
padron["anexo_key"] = padron["ANEXO"].astype(str)

# ubigeo de deserción viene sin cero a la izquierda al leerse como int
desercion_primaria["ubigeo_key"] = desercion_primaria["ubigeo"].astype(str).str.zfill(6)
desercion_secundaria["ubigeo_key"] = desercion_secundaria["ubigeo"].astype(str).str.zfill(6)
padron["ubigeo_key"] = padron["CODGEO"].astype(str).str.zfill(6)

matricula[["cod_mod","cod_mod_key","anexo","anexo_key"]].head()


,cod_mod,cod_mod_key,anexo,anexo_key
15,2212,0002212,0,0
16,2212,0002212,0,0
17,2212,0002212,0,0
18,2212,0002212,0,0
19,2212,0002212,0,0


### <font color=279CF5>2.3 Tabla resumen de cada fuente</font>
Un resumen rápido, previo a integrar, ayuda a detectar problemas de escala o de cobertura.

In [ ]:

print("Resumen 1 - Padrón: escuelas por nivel modular")
display(padron.groupby("D_NIV_MOD").size().reset_index(name="n_escuelas").sort_values("n_escuelas", ascending=False))


Resumen 1 - Padrón: escuelas por nivel modular


,D_NIV_MOD,n_escuelas
14,Primaria,38128
10,Inicial - Jardín,33100
11,Inicial - Programa no escolarizado,17751
15,Secundaria,15214
16,Técnico Productiva,1606
9,Inicial - Cuna-jardín,1588
0,Básica Alternativa-Avanzado,1517
1,Básica Alternativa-Inicial e Intermedio,853
13,Instituto Superior Tecnológico,786
4,Básica Especial-Primaria,454


In [ ]:

print("Resumen 2 - Matrícula 2023: total de estudiantes matriculados por nivel")
resumen_matricula = matricula.groupby("dsc_nivel")["TotalEstudiantes"].sum().reset_index()
display(resumen_matricula)


Resumen 2 - Matrícula 2023: total de estudiantes matriculados por nivel


,dsc_nivel,TotalEstudiantes
0,Primaria,3721852
1,Secundaria,2842831


In [ ]:

print("Resumen 3 - Matrícula 2023: estudiantes por tipo de gestión y nivel")
resumen_gestion = matricula.groupby(["gestion","dsc_nivel"])["TotalEstudiantes"].sum().reset_index()
display(resumen_gestion)


Resumen 3 - Matrícula 2023: estudiantes por tipo de gestión y nivel


,gestion,dsc_nivel,TotalEstudiantes
0,Privada,Primaria,976573
1,Privada,Secundaria,691256
2,Pública de gestión directa,Primaria,2625176
3,Pública de gestión directa,Secundaria,2025101
4,Pública de gestión privada,Primaria,120103
5,Pública de gestión privada,Secundaria,126474


In [ ]:

print("Resumen 4 - Deserción Primaria: desertores y tasa promedio por departamento")
resumen_des_prim = desercion_primaria.groupby("Departamento").agg(
    total_desertores=("desertor","sum"),
    total_matriculados=("denominador","sum"),
    tasa_promedio=("Tasa","mean")
).round(2).sort_values("total_desertores", ascending=False)
display(resumen_des_prim.head(10))


Resumen 4 - Deserción Primaria: desertores y tasa promedio por departamento


,total_desertores,total_matriculados,tasa_promedio
Departamento,,,
LIMA,28543,1064061,1.40
LORETO,4965,176621,3.63
LA LIBERTAD,4325,233970,1.53
CALLAO,3345,114087,3.23
PIURA,3016,257137,0.92
UCAYALI,2826,98150,3.45
LAMBAYEQUE,2094,160132,1.21
JUNIN,1958,156613,0.90
CAJAMARCA,1587,177746,0.92


In [ ]:

print("Resumen 5 - Deserción Secundaria: desertores y tasa promedio por departamento")
resumen_des_sec = desercion_secundaria.groupby("Departamento").agg(
    total_desertores=("desertor","sum"),
    total_matriculados=("denominador","sum"),
    tasa_promedio=("Tasa","mean")
).round(2).sort_values("total_desertores", ascending=False)
display(resumen_des_sec.head(10))


Resumen 5 - Deserción Secundaria: desertores y tasa promedio por departamento


,total_desertores,total_matriculados,tasa_promedio
Departamento,,,
LIMA,22591,812370,2.19
PIURA,6131,195528,3.12
LA LIBERTAD,5991,174599,3.47
LORETO,3905,122023,3.44
CAJAMARCA,3823,139854,2.92
LAMBAYEQUE,3703,117937,3.13
UCAYALI,3385,66211,4.92
ANCASH,3056,106164,2.25
CALLAO,2984,83703,3.70



# <font color=green>Parte 3. `concat` versus `merge`</font>

- Usamos **`concat`** más adelante para apilar `desercion_primaria` y `desercion_secundaria`,
  que comparten exactamente las mismas columnas (Parte 7).
- Usamos **`merge`** para emparejar fuentes con información complementaria sobre una misma unidad
  de análisis usando una llave común: es el caso de matrícula ↔ padrón, y de la base agregada
  ↔ deserción.



# <font color=green>Parte 4. Definir la unidad de análisis y las llaves</font>

Trabajamos en **dos niveles de análisis, en cascada**:

1. **Nivel escuela** (`cod_mod` + `anexo` + nivel educativo): unimos `matricula` con `padron`
   para traerle la geografía (ubigeo/departamento/provincia/distrito) a cada fila de matrícula.
   Llave: `cod_mod_key` + `anexo_key` + `id_nivel` (matrícula) ↔ `cod_mod_key` + `anexo_key` + `NIV_MOD` (padrón).
2. **Nivel distrito x nivel educativo**: agregamos la base anterior por distrito y la
   emparejamos con los archivos de deserción, cuya unidad de análisis ya es el distrito
   (`ubigeo`) para un nivel educativo dado.

Antes de cada `merge`, validamos que la llave no tenga duplicados inesperados.


In [ ]:

# 4.1 Validar duplicados en la llave escuela+nivel dentro de cada fuente
dup_matricula = matricula.duplicated(subset=["cod_mod_key","anexo_key","id_nivel","Edad"]).sum()
dup_padron = padron.duplicated(subset=["cod_mod_key","anexo_key","NIV_MOD"]).sum()

print("Filas duplicadas en matrícula (cod_mod+anexo+nivel+edad):", dup_matricula)
print("Filas duplicadas en padrón (cod_mod+anexo+NIV_MOD):", dup_padron)


Filas duplicadas en matrícula (cod_mod+anexo+nivel+edad): 53777
Filas duplicadas en padrón (cod_mod+anexo+NIV_MOD): 0



`matricula` tiene varias filas por escuela+nivel (una por cada edad, y a veces por tipo de
discapacidad), así que **no** esperamos una fila por escuela: la unidad real de matrícula es
más fina (escuela x nivel x edad). El padrón, en cambio, sí tiene una fila única por
escuela x nivel (`NIV_MOD`), que es justamente lo necesario para usarlo como tabla puente sin
inflar la matrícula al hacer merge.



# <font color=green>Parte 5. Integración auditada (Nivel 1): Matrícula ↔ Padrón</font>

Usamos `how="left"` porque queremos conservar todas las filas de matrícula y solo agregar
la geografía cuando exista coincidencia en el padrón; con `indicator=True` podemos auditar
cuántas filas cruzaron.


In [ ]:

matricula_geo = matricula.merge(
    padron[["cod_mod_key","anexo_key","NIV_MOD","D_NIV_MOD","CODGEO","ubigeo_key",
             "DPTO","PROV","DIST","D_GESTION","DRE_UGEL","AREA_CENSO","DAREACENSO"]],
    left_on=["cod_mod_key","anexo_key","id_nivel"],
    right_on=["cod_mod_key","anexo_key","NIV_MOD"],
    how="left",
    indicator=True
)

conteo_cruce_1 = matricula_geo["_merge"].value_counts().reset_index()
conteo_cruce_1.columns = ["resultado_cruce","cantidad"]
display(conteo_cruce_1)


,resultado_cruce,cantidad
0,both,392095
1,left_only,1010
2,right_only,0


In [ ]:

fig1 = px.bar(
    conteo_cruce_1,
    x="resultado_cruce",
    y="cantidad",
    text="cantidad",
    title="Nivel 1 - Cruce Matrícula ↔ Padrón (llave: escuela + nivel)"
)
fig1.show()


Intepretación: 392,095 filas son las filas de matrícula que sí encontraron su escuela correspondiente en el padrón. Es decir, se les pudo asignar distrito, departamento, provincia, ubigeo, etc. Esta es la barra que va a dominar el gráfico visualmente casi toda la matrícula cruzó bien.

1,010 filas son filas de matrícula cuya combinación cod_mod + anexo + id_nivel no apareció en el padrón. Es decir, hay estudiantes matriculados en una escuela+nivel que el padrón no reconoce con esa misma llave probablemente por errores de digitación en el código modular, escuelas que cerraron/cambiaron de código entre la fecha del padrón y la matrícula, o inconsistencias menores entre ambos registros administrativos.

In [ ]:

print("Escuelas de matrícula que NO encontraron par en el padrón (muestra):")
no_cruzan_1 = matricula_geo[matricula_geo["_merge"] == "left_only"]
print("Filas sin geografía:", no_cruzan_1.shape[0], f"({no_cruzan_1.shape[0]/matricula_geo.shape[0]:.2%} del total)")
display(no_cruzan_1[["cod_mod","anexo","Nombre","dsc_nivel"]].drop_duplicates().head(10))


Escuelas de matrícula que NO encontraron par en el padrón (muestra):
Filas sin geografía: 1010 (0.26% del total)


,cod_mod,anexo,Nombre,dsc_nivel
20988,244384,0,72037,Primaria
61330,331769,0,NUESTRA SEÑORA DEL CARMELO,Primaria
131569,502245,0,JESUS NUESTRA ALEGRIA,Primaria
150269,567727,0,SAN FRANCISCO SOLANO,Primaria
181556,664649,0,EL DIVINO HACEDOR,Primaria
199993,732289,0,HAPPY PLACE,Primaria
201333,737445,0,PADRE JOSE SAFARICK,Primaria
206973,762732,0,MARIA MONTESSORI,Primaria
210057,778134,0,MI JESUS,Secundaria
210074,778183,0,RICARDO PALMA DE CHOSICA,Primaria



**Precaución:** las filas sin geografía (`left_only`) no se pueden ubicar en un distrito y por
lo tanto **no podrán cruzar** con los archivos de deserción en el siguiente paso. Se conservan
en `matricula_geo` para no perder información de matrícula, pero se excluirán al construir la
base agregada por distrito.



# <font color=green>Parte 6. Agregar matrícula 2023 a nivel distrito x nivel</font>

La deserción se define como *matriculado en 2023 que no volvió a matricularse en 2024*, y su
`denominador` es esa matrícula 2023. Para poder cruzarla, agregamos la matrícula (ya
georreferenciada) a nivel distrito x nivel educativo.


La matrícula tiene varias filas por distrito, ya que registra cada escuela y edad, mientras que la deserción tiene una sola fila por distrito y nivel. Por eso, primero se agrupa la matrícula para obtener un solo registro por distrito y nivel. Así, el merge se realizará correctamente y se pueden comparar ambas bases sin repetir los datos de deserción.

In [ ]:

base_distrital = (
    matricula_geo[matricula_geo["_merge"] == "both"]
    .groupby(["ubigeo_key","DPTO","PROV","DIST","dsc_nivel"], as_index=False)
    .agg(
        total_estudiantes_2023=("TotalEstudiantes","sum"),
        total_mujeres_2023=("Mujer","sum"),
        total_hombres_2023=("Hombre","sum"),
        total_aprobados_2023=("Aprobado","sum"),
        total_desaprobados_2023=("Desaprobado","sum"),
        total_retirados_2023=("Retirado","sum"),
        n_escuelas_2023=("cod_mod_key","nunique"),
    )
)

print("Base agregada a nivel distrito x nivel (matrícula 2023):", base_distrital.shape)
base_distrital.head()


Base agregada a nivel distrito x nivel (matrícula 2023): (3730, 12)


,ubigeo_key,DPTO,PROV,DIST,dsc_nivel,total_estudiantes_2023,total_mujeres_2023,total_hombres_2023,total_aprobados_2023,total_desaprobados_2023,total_retirados_2023,n_escuelas_2023
0,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,Primaria,4146,2110,2036,4100,38,6,18
1,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,Secundaria,3421,1726,1695,3294,104,21,9
2,010102,AMAZONAS,CHACHAPOYAS,ASUNCION,Primaria,12,5,7,12,0,0,3
3,010103,AMAZONAS,CHACHAPOYAS,BALSAS,Primaria,168,84,84,161,7,0,10
4,010103,AMAZONAS,CHACHAPOYAS,BALSAS,Secundaria,125,53,72,116,8,1,2



# <font color=green>Parte 7. Preparar e integrar los archivos de deserción</font>

Ambos archivos de deserción comparten estructura (`ubigeo`, `Departamento`, `Provincia`,
`Distrito`, `desertor`, `denominador`, `Tasa`), pero cada uno corresponde a **un nivel
educativo distinto** (Primaria / Secundaria). Recordemos: `denominador` = matriculados **2023**
y `desertor` = de ellos, quienes no se volvieron a matricular en **2024**. Apilamos ambos
archivos con `concat`, agregando una columna `dsc_nivel`, para luego cruzarlos con
`base_distrital` (que ya está construida solo con matrícula 2023) usando la llave
`ubigeo + nivel`.


Se usa concat porque los archivos de deserción de Primaria y Secundaria tienen las mismas columnas y representan el mismo tipo de información.

concat permite unirlos verticalmente, es decir, colocar las filas de Primaria debajo de las filas de Secundaria.

In [ ]:

desercion_primaria["dsc_nivel"] = "Primaria"
desercion_secundaria["dsc_nivel"] = "Secundaria"

desercion = pd.concat([desercion_primaria, desercion_secundaria], ignore_index=True)
desercion = desercion.rename(columns={
    "desertor": "n_desertores",
    "denominador": "n_matriculados_base_desercion",
    "Tasa": "tasa_desercion_pct"
})

print("Deserción (Primaria + Secundaria) concatenada:", desercion.shape)
desercion.head()


Deserción (Primaria + Secundaria) concatenada: (3736, 9)


,ubigeo,Departamento,Provincia,Distrito,n_desertores,n_matriculados_base_desercion,tasa_desercion_pct,ubigeo_key,dsc_nivel
0,10101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,17,4146,0.410034,010101,Primaria
1,10102,AMAZONAS,CHACHAPOYAS,ASUNCION,0,12,0.000000,010102,Primaria
2,10103,AMAZONAS,CHACHAPOYAS,BALSAS,0,168,0.000000,010103,Primaria
3,10104,AMAZONAS,CHACHAPOYAS,CHETO,0,44,0.000000,010104,Primaria
4,10105,AMAZONAS,CHACHAPOYAS,CHILIQUIN,3,62,4.838710,010105,Primaria


Se valida que ubigeo + dsc_nivel no esté duplicado en la base de deserción.

In [ ]:

# Validar duplicados en la llave ubigeo + nivel
dup_desercion = desercion.duplicated(subset=["ubigeo_key","dsc_nivel"]).sum()
print("Duplicados en deserción (ubigeo+nivel):", dup_desercion)


Duplicados en deserción (ubigeo+nivel): 0


### <font color=279CF5>7.1 Integración auditada (Nivel 2): Base distrital ↔ Deserción</font>

In [ ]:
base_validacion_2 = base_distrital.merge(
    desercion[["ubigeo_key","dsc_nivel","n_desertores","n_matriculados_base_desercion","tasa_desercion_pct"]],
    on=["ubigeo_key","dsc_nivel"],
    how="outer",
    indicator=True,
    suffixes=("_matricula","_desercion")
)

conteo_cruce_2 = base_validacion_2["_merge"].value_counts().reset_index()
conteo_cruce_2.columns = ["resultado_cruce","cantidad"]
conteo_cruce_2["porcentaje"] = 100 * conteo_cruce_2["cantidad"] / conteo_cruce_2["cantidad"].sum()
display(conteo_cruce_2)

,resultado_cruce,cantidad,porcentaje
0,both,3730,99.8394
1,right_only,6,0.1606
2,left_only,0,0.0000


In [ ]:

fig2 = px.bar(
    conteo_cruce_2,
    x="resultado_cruce",
    y="cantidad",
    text="cantidad",
    title="Nivel 2 - Cruce Base distrital (Matrícula+Padrón) ↔ Deserción (llave: ubigeo + nivel)"
)
fig2.show()


3,730 filas de las 3,736 combinaciones distrito×nivel involucradas): distritos×nivel que tienen tanto matrícula agregada como dato oficial de deserción. Esta es tu base final de análisis.
right_only

6 filas de distritos×nivel que sí aparecen en el archivo de deserción, pero no encontraron matrícula agregada del lado izquierdo. Es decir, hay un dato de deserción reportado para ese distrito, pero nuestra matrícula (después del primer merge con el padrón) no logró cubrir ese distrito.

no hay ningún distrito×nivel con matrícula agregada que se haya quedado sin dato de deserción, toda la matrícula que llegó a este paso encontró su contraparte de deserción.

In [ ]:

print("Registros que no cruzan entre la base distrital y deserción:")
no_cruzan_2 = base_validacion_2[base_validacion_2["_merge"] != "both"]
display(no_cruzan_2[["ubigeo_key","DPTO","DIST","dsc_nivel","_merge"]].head(15))


Registros que no cruzan entre la base distrital y deserción:


,ubigeo_key,DPTO,DIST,dsc_nivel,_merge
1534,080916,NaN,NaN,Primaria,right_only
1535,080916,NaN,NaN,Secundaria,right_only
1536,080917,NaN,NaN,Primaria,right_only
1537,080917,NaN,NaN,Secundaria,right_only
1538,080918,NaN,NaN,Primaria,right_only
1539,080918,NaN,NaN,Secundaria,right_only


 `left_only` son distritos con matrícula pero sin dato de deserción reportado
(posiblemente por secreto estadístico en distritos muy pequeños); `right_only` son distritos
con deserción reportada pero cuya matrícula no logró georreferenciarse en el Nivel 1 (por
ejemplo, escuelas sin coincidencia exacta en el padrón). Ambos casos deben excluirse del
análisis de asociación matrícula-deserción, pero se documentan aquí en vez de descartarse en
silencio.



# <font color=green>Parte 8. Construir la base integrada final para análisis</font>

Nos quedamos con los distritos x nivel que **sí cruzan** en ambos niveles de integración
(`_merge == "both"`), que es la base con la que podemos estudiar la relación entre indicadores
de matrícula/trayectoria y la deserción escolar.


In [ ]:

base_final = base_validacion_2[base_validacion_2["_merge"] == "both"].drop(columns="_merge").copy()

print("Tamaño de la base integrada final:", base_final.shape)
base_final.head()


Tamaño de la base integrada final: (3730, 15)


,ubigeo_key,DPTO,PROV,DIST,dsc_nivel,total_estudiantes_2023,total_mujeres_2023,total_hombres_2023,total_aprobados_2023,total_desaprobados_2023,total_retirados_2023,n_escuelas_2023,n_desertores,n_matriculados_base_desercion,tasa_desercion_pct
0,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,Primaria,4146.0,2110.0,2036.0,4100.0,38.0,6.0,18.0,17,4146,0.410034
1,010101,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,Secundaria,3421.0,1726.0,1695.0,3294.0,104.0,21.0,9.0,42,3421,1.227711
2,010102,AMAZONAS,CHACHAPOYAS,ASUNCION,Primaria,12.0,5.0,7.0,12.0,0.0,0.0,3.0,0,12,0.000000
3,010103,AMAZONAS,CHACHAPOYAS,BALSAS,Primaria,168.0,84.0,84.0,161.0,7.0,0.0,10.0,0,168,0.000000
4,010103,AMAZONAS,CHACHAPOYAS,BALSAS,Secundaria,125.0,53.0,72.0,116.0,8.0,1.0,2.0,5,125,4.000000


**Diccionario**

In [ ]:
descripciones = {
    "ubigeo": "Código de ubicación geográfica del distrito.",
    "Departamento": "Departamento del Perú.",
    "Provincia": "Provincia del Perú.",
    "Distrito": "Distrito del Perú.",
    "nivel": "Tipo de nivel educativo: Primaria o Secundaria.",
    "total_estudiantes_2023": "Total de estudiantes matriculados en 2023.",
    "total_mujeres_2023": "Total de mujeres matriculadas en 2023.",
    "total_hombres_2023": "Total de hombres matriculados en 2023.",
    "total_aprobados_2023": "Total de estudiantes aprobados en 2023.",
    "total_desaprobados_2023": "Total de estudiantes desaprobados en 2023.",
    "total_retirados_2023": "Total de estudiantes retirados en 2023.",
    "n_escuelas_2023": "Número de escuelas en 2023.",
    "desertores": "Número de desertores según la base de Tasa y Desertores de Educación Primaria y Secundaria 2023.",
    "n_matriculados_base_desercion": "Número de estudiantes matriculados en 2023 utilizado como base para calcular la tasa de deserción.",
    "tasa_desercion_pct": "Tasa de deserción expresada en porcentaje."
}

tipos_variable = {
    "ubigeo": "Cualitativa nominal",
    "Departamento": "Cualitativa nominal",
    "Provincia": "Cualitativa nominal",
    "Distrito": "Cualitativa nominal",
    "nivel": "Cualitativa nominal",
    "total_estudiantes_2023": "Cuantitativa discreta",
    "total_mujeres_2023": "Cuantitativa discreta",
    "total_hombres_2023": "Cuantitativa discreta",
    "total_aprobados_2023": "Cuantitativa discreta",
    "total_desaprobados_2023": "Cuantitativa discreta",
    "total_retirados_2023": "Cuantitativa discreta",
    "n_escuelas_2023": "Cuantitativa discreta",
    "desertores": "Cuantitativa discreta",
    "n_matriculados_base_desercion": "Cuantitativa discreta",
    "tasa_desercion_pct": "Cuantitativa continua"
}

diccionario_base_final = pd.DataFrame({
    "Nombre_Variable": descripciones.keys(),
    "Descripcion": descripciones.values(),
    "Tipo_Variable": [tipos_variable[col] for col in descripciones.keys()],
     "Tipo_Dato": [str(base_final[col].dtype) for col in base_final.columns]
})

display(diccionario_base_final)

,Nombre_Variable,Descripcion,Tipo_Variable,Tipo_Dato
0,ubigeo,Código de ubicación geográfica del distrito.,Cualitativa nominal,object
1,Departamento,Departamento del Perú.,Cualitativa nominal,object
2,Provincia,Provincia del Perú.,Cualitativa nominal,object
3,Distrito,Distrito del Perú.,Cualitativa nominal,object
4,nivel,Tipo de nivel educativo: Primaria o Secundaria.,Cualitativa nominal,object
5,total_estudiantes_2023,Total de estudiantes matriculados en 2023.,Cuantitativa discreta,float64
6,total_mujeres_2023,Total de mujeres matriculadas en 2023.,Cuantitativa discreta,float64
7,total_hombres_2023,Total de hombres matriculados en 2023.,Cuantitativa discreta,float64
8,total_aprobados_2023,Total de estudiantes aprobados en 2023.,Cuantitativa discreta,float64
9,total_desaprobados_2023,Total de estudiantes desaprobados en 2023.,Cuantitativa discreta,float64



### <font color=279CF5>8.1 Validación cruzada: ¿el denominador de deserción coincide con nuestra propia matrícula 2023?</font>

Como `n_matriculados_base_desercion` (el `denominador` del archivo de deserción) debería ser,
en teoría, la misma matrícula 2023 que calculamos de forma independiente en la Parte 6
(`total_estudiantes_2023`), comparar ambas columnas es una buena prueba de consistencia entre
fuentes: si difieren mucho, hay un problema de cobertura o de definición que hay que investigar
antes de confiar en la tasa de deserción a nivel distrito.


In [ ]:

# Cálculo puntual para esta validación (no se agrega a base_final: es solo diagnóstico)
diferencia_pct = (
    100 * (base_final["n_matriculados_base_desercion"] - base_final["total_estudiantes_2023"])
    / base_final["n_matriculados_base_desercion"]
)

print("Estadísticos de la diferencia (denominador oficial - matrícula 2023 calculada):")
display(diferencia_pct.describe().round(2))

print("\nDistritos x nivel con mayor discrepancia (top 10 en valor absoluto):")
tabla_diferencia = base_final[["DPTO","DIST","dsc_nivel","total_estudiantes_2023","n_matriculados_base_desercion"]].copy()
tabla_diferencia["diferencia_pct"] = diferencia_pct
display(tabla_diferencia.reindex(diferencia_pct.abs().sort_values(ascending=False).index).head(10))


Estadísticos de la diferencia (denominador oficial - matrícula 2023 calculada):


,0
count,3730.00
mean,-0.01
std,1.69
min,-55.43
25%,0.00
50%,0.00
75%,0.00
max,32.89



Distritos x nivel con mayor discrepancia (top 10 en valor absoluto):


,DPTO,DIST,dsc_nivel,total_estudiantes_2023,n_matriculados_base_desercion,diferencia_pct
1517,CUSCO,KIMBIRI,Secundaria,1545.0,994,-55.432596
1516,CUSCO,KIMBIRI,Primaria,2492.0,1688,-47.630332
2337,LA LIBERTAD,MAGDALENA DE CAO,Primaria,153.0,228,32.894737
1522,CUSCO,PICHARI,Primaria,5066.0,3981,-27.254459
1523,CUSCO,PICHARI,Secundaria,3398.0,2782,-22.142344
2342,LA LIBERTAD,RAZURI,Secundaria,704.0,589,-19.524618
2341,LA LIBERTAD,RAZURI,Primaria,1104.0,950,-16.210526
1529,CUSCO,VILLA KINTIARINA,Secundaria,281.0,330,14.848485
868,AYACUCHO,CARMEN ALTO,Secundaria,2243.0,1974,-13.627153
2742,LIMA,LAHUAYTAMBO,Primaria,53.0,61,13.114754


Se comprueba que ambas columnas representan el mismo concepto: estudiantes matriculados en 2023 por distrito y nivel. Una proviene del dato oficial de deserción y la otra la calculamos a partir de la matrícula. Al compararlas y obtener una diferencia mediana de 0%, confirmamos que nuestra integración es consistente.

La integración tiene una tasa de cruce del 99.7% en ambos merges, cero duplicados y cero nulos en la base final. Esto no significa que no haya errores — documentamos 0.3% de registros que no cruzaron, y algunos distritos rurales (como Kimbiri y Pichari) muestran discrepancias notables al validar contra el denominador oficial de deserción — pero para la gran mayoría del país, la base integrada representa fielmente la matrícula y deserción reales.


# <font color=green>Parte 9. Valores nulos y outliers en la base integrada</font>


In [ ]:

print("Valores nulos en la base final:")
display(base_final.isna().sum())


Valores nulos en la base final:


,0
ubigeo_key,0
DPTO,0
PROV,0
DIST,0
dsc_nivel,0
total_estudiantes_2023,0
total_mujeres_2023,0
total_hombres_2023,0
total_aprobados_2023,0
total_desaprobados_2023,0


In [ ]:

# Outliers con la regla de rango intercuartílico (IQR)
def resumen_outliers_iqr(serie, nombre):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_outliers = ((serie < lim_inf) | (serie > lim_sup)).sum()
    return {
        "variable": nombre, "Q1": round(q1,2), "Q3": round(q3,2),
        "limite_inferior": round(lim_inf,2), "limite_superior": round(lim_sup,2),
        "n_outliers": n_outliers, "pct_outliers": round(100*n_outliers/len(serie),2)
    }

variables_numericas = ["total_estudiantes_2023","n_desertores","tasa_desercion_pct","n_escuelas_2023"]
resumen_out = pd.DataFrame([resumen_outliers_iqr(base_final[v], v) for v in variables_numericas])
display(resumen_out)


,variable,Q1,Q3,limite_inferior,limite_superior,n_outliers,pct_outliers
0,total_estudiantes_2023,140.25,1186.00,-1428.38,2754.62,470,12.60
1,n_desertores,0.00,18.00,-27.00,45.00,539,14.45
2,tasa_desercion_pct,0.00,2.17,-3.26,5.43,145,3.89
3,n_escuelas_2023,3.00,15.00,-15.00,33.00,356,9.54


In [ ]:

fig3 = px.box(
    base_final,
    x="dsc_nivel",
    y="tasa_desercion_pct",
    points="outliers",
    color="dsc_nivel",
    title="Distribución de la tasa de deserción por nivel educativo (outliers marcados)"
)
fig3.show()


**Interpretación:**

Primaria: presenta una tasa de deserción generalmente baja, con una mediana cercana a 0,3%. La mayor parte de los valores se concentra aproximadamente entre 0% y 1%.

Secundaria: tiene una tasa de deserción considerablemente mayor, con una mediana cercana a 1% y la mayoría de los datos entre aproximadamente 0,6% y 2,2%.

Dispersión: Secundaria muestra valores centrales más elevados y una distribución relativamente más amplia, aunque Primaria presenta numerosos valores extremos.

Valores atípicos (outliers): ambos niveles tienen observaciones muy alejadas del resto. En Primaria se observan valores que superan el 10%, mientras que en Secundaria también aparecen casos extremos de hasta aproximadamente 11%.

Asimetría: las dos distribuciones presentan asimetría positiva, debido a que existen algunos casos con tasas de deserción muy altas.


**Nota sobre outliers:** distritos con `total_estudiantes_2023` o `n_desertores` muy altos suelen
corresponder a distritos urbanos grandes (p. ej. capitales de provincia), no a errores de
datos; antes de eliminarlos hay que revisar si el outlier es un valor genuino (heterogeneidad
real entre distritos) o un problema de calidad de datos.


# <font color=green>Parte 11. Visualizaciones</font>

### <font color=279CF5>11.1 Barras: total de estudiantes matriculados por gestión y nivel (2023)</font>

In [ ]:

fig4 = px.bar(
    resumen_gestion,
    x="gestion",
    y="TotalEstudiantes",
    color="dsc_nivel",
    barmode="group",
    text="TotalEstudiantes",
    title="Estudiantes matriculados por tipo de gestión y nivel (2023)"
)
fig4.show()


**Interpretación:** En 2023, la mayor cantidad de estudiantes matriculados se concentra en las instituciones públicas de gestión directa, con 1 287 271 estudiantes en Primaria y 440 939 en Secundaria. En la gestión privada, se registran 104 104 matriculados en Primaria y 59 914 en Secundaria. Por otro lado, las instituciones públicas de gestión privada presentan 36 982 estudiantes en Primaria y 35 378 en Secundaria. En general, la gestión pública directa concentra la mayor cantidad de estudiantes matriculados en ambos niveles educativos.

### <font color=279CF5>11.2 Histograma: distribución de la tasa de deserción</font>

In [ ]:

fig5 = px.histogram(
    base_final,
    x="tasa_desercion_pct",
    color="dsc_nivel",
    nbins=40,
    marginal="box",
    title="Distribución de la tasa de deserción (%) por distrito y nivel"
)
fig5.show()


**Interpretación:** La tasa de deserción es generalmente baja en ambos niveles, ya que la mayoría de los distritos presenta tasas cercanas a 0%–2%. Sin embargo, Secundaria presenta una distribución más alta y dispersa que Primaria, con varios distritos que superan el 4% y algunos valores extremos cercanos al 10%–11%. En Primaria también existen valores atípico. En general, la deserción es mayor en Secundaria que en Primaria y presenta mayor variabilidad entre distritos.

### <font color=279CF5>11.3 Dispersión: total de estudiantes vs. tasa de deserción por distrito</font>

In [ ]:

fig6 = px.scatter(
    base_final,
    x="total_estudiantes_2023",
    y="tasa_desercion_pct",
    color="dsc_nivel",
    size="n_desertores",
    hover_data=["DPTO","DIST"],
    log_x=True,
    title="Matrícula total vs. tasa de deserción, por distrito (tamaño = n° desertores)"
)
fig6.show()


**Interpretación:** El gráfico muestra que la mayoría de los distritos tiene una tasa de deserción menor al 2%, independientemente del número de estudiantes matriculados. Sin embargo, se observan algunos distritos con tasas superiores al 4%, principalmente en Secundaria. Además, los círculos más grandes representan un mayor número de desertores. En general, no se observa una relación clara entre el número de estudiantes matriculados y la tasa de deserción, ya que existen distritos con muchos estudiantes y tasas bajas, así como otros con tasas elevadas.

# <font color=green>Parte 12. Del gráfico al hallazgo</font>

In [ ]:

hallazgos = pd.DataFrame({
    "grafico": [
        "Resultado de integración (Nivel 1 y 2)",
        "Estudiantes por gestión y nivel",
        "Retiro intra-año vs. tasa oficial de deserción",
        "Distribución de la tasa de deserción",
        "Matrícula vs. tasa de deserción"
    ],
    "observacion": ["", "", "", "", ""],
    "posible_hallazgo_o_hipotesis": ["", "", "", "", ""],
    "precaucion": [
        "Revisar los distritos left_only/right_only antes de excluirlos del análisis.",
        "La gestión pública concentra la mayoría de estudiantes; comparar tasas, no solo totales.",
        "Correlación baja/moderada esperada: son conceptos distintos (retiro intra-año vs. no reinscripción).",
        "La deserción es de cohorte (2023→2024): revisar la validación de denominadores (8.1) antes de generalizar.",
        "Relación visual no implica causalidad; considerar variables de contexto (ruralidad, pobreza)."
    ]
})
hallazgos


,grafico,observacion,posible_hallazgo_o_hipotesis,precaucion
0,Resultado de integración (Nivel 1 y 2),,,Revisar los distritos left_only/right_only ant...
1,Estudiantes por gestión y nivel,,,La gestión pública concentra la mayoría de est...
2,Retiro intra-año vs. tasa oficial de deserción,,,Correlación baja/moderada esperada: son concep...
3,Distribución de la tasa de deserción,,,La deserción es de cohorte (2023→2024): revisa...
4,Matrícula vs. tasa de deserción,,,Relación visual no implica causalidad; conside...



# <font color=green>Parte 13. Exportar la base integrada</font>


In [ ]:

base_final.to_csv("base_integrada_educacion.csv", index=False, encoding="utf-8-sig")
print("Archivo generado: base_integrada_educacion.csv")
print("Filas:", base_final.shape[0], "| Columnas:", base_final.shape[1])


Archivo generado: base_integrada_educacion.csv
Filas: 3730 | Columnas: 15


In [ ]:
with pd.ExcelWriter("base_integrada_educacion.xlsx", engine="openpyxl") as writer:

    #Base integrada
    base_final.to_excel(
        writer,
        sheet_name="Base_Final",
        index=False
    )

    #Diccionario de variables
    diccionario_base_final.to_excel(
        writer,
        sheet_name="Diccionario",
        index=False
    )

print("Archivo generado: base_integrada_educacion.xlsx")
print("Filas:", base_final.shape[0], "| Columnas:", base_final.shape[1])

Archivo generado: base_integrada_educacion.xlsx
Filas: 3730 | Columnas: 15
